# ESS Round 4 data curation: all-adult outputs with and without official ESS weights

This notebook reproduces the deterministic ESS Round 4 belief construction based on Jochem van Noord's supplied `Data cleaning_ESS4.R` script and creates the respondent-level datasets used in the present project.

The principal outputs:

- retain respondents aged 18 or older;
- also retain respondents whose age is missing;
- **do not exclude respondents because of the number of missing constructed beliefs**;
- leave missing belief values as missing (`NaN`) rather than imputing them;
- record the number of missing and available beliefs for every respondent;
- retain a Boolean `cca_missingness_eligible` flag identifying respondents who satisfy Van Noord et al.'s historical CCA criterion of no more than two missing beliefs;
- create separate datasets with and without the four official ESS survey-weight columns.

The historical CCA-compatible subset is reconstructed only for later respondent-level validation against `df_ESS4.RData`. It is not used as the principal output sample.

## Step 1 — Locate the project, import the shared code, and define paths

Run this notebook from either the project root or the `notebooks/` folder. The following cell locates the folder containing `data/`, `notebooks/`, and `src/`, adds it to Python's import path, imports the shared curation functions, and defines the ESS4 input and output paths.

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from IPython.display import display


def locate_project_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if all(
            (candidate / folder).is_dir()
            for folder in ("data", "notebooks", "src")
        ):
            return candidate
    raise FileNotFoundError(
        "Could not locate the project root containing "
        "data/, notebooks/, and src/."
    )


PROJECT_ROOT = locate_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src import ess4_config as config
from src.ess_curation_common import (
    apply_analysis_sample_rule,
    construct_belief_variables,
    require_columns,
    split_weighted_and_unweighted,
    summarise_beliefs,
    valid_range,
    validate_weight_split,
)

RAW_DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / config.RAW_DATA_FOLDER
    / config.RAW_DATA_FILENAME
)
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

WITHOUT_WEIGHTS_PATH = (
    PROCESSED_DIR / config.OUTPUT_FILENAMES["without_weights"]
)
WITH_WEIGHTS_PATH = (
    PROCESSED_DIR / config.OUTPUT_FILENAMES["with_weights"]
)

print("Project root:", PROJECT_ROOT)
print("Raw ESS4 file:", RAW_DATA_PATH)
print("Processed-data folder:", PROCESSED_DIR)
print("Without-weights output:", WITHOUT_WEIGHTS_PATH.name)
print("With-weights output:", WITH_WEIGHTS_PATH.name)

Project root: /Users/karan/Desktop/SSM-MERC/polarization/Github/C72H-SimPol/data_curation
Raw ESS4 file: /Users/karan/Desktop/SSM-MERC/polarization/Github/C72H-SimPol/data_curation/data/ESS4e04_6/ESS4e04_6.csv
Processed-data folder: /Users/karan/Desktop/SSM-MERC/polarization/Github/C72H-SimPol/data_curation/data/processed
Without-weights output: ess4_beliefs_all_adults_without_weights.csv
With-weights output: ess4_beliefs_all_adults_with_weights.csv


## Step 2 — Load the raw ESS Round 4 file and verify the required inputs

No rows or values are changed in this step. The checks confirm the expected raw sample size, country count, required metadata variables, constituent belief items, and four official ESS weight columns.

In [2]:
raw = pd.read_csv(RAW_DATA_PATH, low_memory=False)

require_columns(
    raw,
    config.REQUIRED_RAW_COLUMNS,
    context="raw ESS Round 4 file",
)

assert len(raw) == config.EXPECTED_RAW_N, (
    f"Expected {config.EXPECTED_RAW_N:,} raw rows, "
    f"found {len(raw):,}."
)
assert (
    raw["cntry"].nunique(dropna=True)
    == config.EXPECTED_RAW_COUNTRIES
)

weight_missing_raw = (
    raw.loc[:, config.WEIGHT_COLUMNS].isna().sum()
)
assert (weight_missing_raw == 0).all(), (
    "Unexpected missing values in raw ESS weights: "
    f"{weight_missing_raw.to_dict()}"
)

print(f"Raw respondents: {len(raw):,}")
print(f"Raw columns: {raw.shape[1]:,}")
print(
    "Countries:",
    raw["cntry"].nunique(dropna=True),
)
print(
    "Official ESS weight columns:",
    list(config.WEIGHT_COLUMNS),
)
display(
    weight_missing_raw
    .rename("missing_values")
    .to_frame()
)

Raw respondents: 56,752
Raw columns: 674
Countries: 29
Official ESS weight columns: ['dweight', 'pspwght', 'pweight', 'anweight']


,missing_values
dweight,0
pspwght,0
pweight,0
anweight,0


## Step 3 — Construct identifiers, demographics, and official ESS weights

The metadata follow the common project schema.

- `ess_row_id` records the original row position in the raw ESS4 CSV, starting at 1.
- `ess_unique_id` combines the country code and ESS respondent ID.
- Invalid or non-substantive ESS values outside each variable's valid response range become missing.
- `education_3cat` follows Van Noord's Round 4 recoding of `edulvla`: values 1–2 are lower education, 3–4 are middle education, and 5 is higher education.
- `urbanization = 6 - domicil`, so higher values indicate more urban surroundings.
- The four official ESS weight variables are copied directly and are not transformed.

In [3]:
metadata = pd.DataFrame(index=raw.index)

metadata["ess_row_id"] = np.arange(1, len(raw) + 1)
metadata["idno"] = pd.to_numeric(
    raw["idno"],
    errors="coerce",
).astype("Int64")
metadata["cntry"] = (
    raw["cntry"].astype("string").str.strip()
)
metadata["country_name"] = (
    metadata["cntry"].map(config.COUNTRY_LABELS)
)
metadata["ess_unique_id"] = (
    metadata["cntry"]
    + "_"
    + metadata["idno"].astype("string")
)

for column in (
    "agea",
    "gndr",
    "edulvla",
    "hinctnta",
    "rlgblg",
    "blgetmg",
):
    minimum, maximum = config.METADATA_VALID_RANGES[column]
    metadata[column] = valid_range(
        raw[column],
        minimum,
        maximum,
    )

metadata["education_3cat"] = np.nan
for category, source_values in (
    config.EDUCATION_3CAT_MAP.items()
):
    metadata.loc[
        metadata["edulvla"].isin(source_values),
        "education_3cat",
    ] = category

minimum, maximum = (
    config.METADATA_VALID_RANGES["domicil"]
)
domicil_clean = valid_range(
    raw["domicil"],
    minimum,
    maximum,
)
metadata["urbanization"] = (
    config.URBANIZATION_REVERSE_CONSTANT
    - domicil_clean
)

for column in config.WEIGHT_COLUMNS:
    metadata[column] = pd.to_numeric(
        raw[column],
        errors="coerce",
    )

metadata = metadata.loc[
    :, config.METADATA_COLUMNS_WITH_WEIGHTS
]

assert metadata["country_name"].notna().all()
assert metadata["ess_unique_id"].notna().all()
assert metadata["ess_unique_id"].is_unique
assert (
    metadata.loc[:, config.WEIGHT_COLUMNS]
    .notna()
    .all()
    .all()
)

print("Metadata shape:", metadata.shape)
display(metadata.head())

Metadata shape: (56752, 17)


,ess_row_id,idno,cntry,country_name,ess_unique_id,agea,gndr,edulvla,education_3cat,hinctnta,rlgblg,urbanization,blgetmg,dweight,pspwght,pweight,anweight
0,1,10202,BE,Belgium,BE_10202,36.0,1.0,5.0,3.0,4.0,2.0,5.0,2.0,1.0074,0.823223,0.503773,0.414718
1,2,10203,BE,Belgium,BE_10203,26.0,2.0,5.0,3.0,7.0,2.0,5.0,2.0,1.0074,0.798609,0.503773,0.402318
2,3,10207,BE,Belgium,BE_10207,69.0,1.0,5.0,3.0,10.0,1.0,5.0,2.0,1.0074,0.778020,0.503773,0.391946
3,4,10208,BE,Belgium,BE_10208,77.0,2.0,5.0,3.0,7.0,1.0,5.0,2.0,1.0074,0.777735,0.503773,0.391802
4,5,10302,BE,Belgium,BE_10302,27.0,1.0,3.0,2.0,7.0,2.0,5.0,2.0,1.0074,0.960960,0.503773,0.484106


## Step 4 — Construct the 19 ESS Round 4 belief variables

The round-specific item definitions, valid response ranges, coding directions, and special recodes are stored in `src/ess4_config.py`.

Important Round 4 details include:

- `txearn` is recoded as 1 → 2, 2 → 1, and 3 → 3 before rescaling;
- response 4 is treated as missing for `earnpen` and `earnueb`;
- `dfincac` is not included in the final anti-egalitarianism scale, matching the supplied R script;
- all beliefs are placed on a 0–1 scale;
- multi-item beliefs use a strict row mean, so a missing constituent item makes that constructed belief missing.

No missing belief value is imputed.

In [4]:
beliefs, coded_items = construct_belief_variables(
    raw,
    config.BELIEF_MAP,
    config.ITEM_CODING,
    item_value_recoding=config.ITEM_VALUE_RECODING,
    item_values_forced_missing=(
        config.ITEM_VALUES_FORCED_MISSING
    ),
)

assert beliefs.shape == (
    len(raw),
    config.EXPECTED_BELIEF_COUNT,
)
assert list(beliefs.columns) == list(
    config.BELIEF_COLUMNS
)
assert list(coded_items.columns) == list(
    config.BELIEF_ITEM_COLUMNS
)

belief_minimum = beliefs.min(
    skipna=True
).min()
belief_maximum = beliefs.max(
    skipna=True
).max()

assert belief_minimum >= 0.0
assert belief_maximum <= 1.0

print(
    "Constructed belief variables:",
    beliefs.shape[1],
)
print(
    "Cleaned constituent ESS items:",
    coded_items.shape[1],
)
print(
    f"Observed belief range: "
    f"[{belief_minimum:.3f}, {belief_maximum:.3f}]"
)
display(beliefs.head())

Constructed belief variables: 19
Cleaned constituent ESS items: 39
Observed belief range: [0.000, 1.000]


,left_right_identification,gender_inequality,anti_lgbt,euroscepticism,anti_immigration,anti_egalitarianism,benefits_harm_economy,benefits_harm_society,welfare_chauvinism,anti_economic_interventionism,harsh_sentences,anti_militant_democracy,no_science_environment_solution,anti_government_spending,regressive_taxes,regressive_benefits,age_prejudice,authoritarianism,anti_libertarianism
0,0.7,0.50,0.75,0.8,1.000000,0.250,0.875,0.250,0.75,0.316667,0.25,0.75,0.25,0.5,0.5,0.0,0.6,0.48,0.40
1,0.6,0.75,0.25,0.4,0.444444,0.250,0.375,0.250,0.50,0.300000,0.75,0.25,0.50,0.5,0.5,0.0,0.3,0.60,0.44
2,0.8,0.75,0.00,0.2,0.333333,0.750,0.750,0.625,0.00,0.550000,0.50,0.75,0.75,0.5,0.0,0.0,0.6,0.56,0.20
3,0.6,0.75,0.50,0.4,0.555556,0.250,0.625,0.250,0.50,0.500000,0.50,0.25,0.25,0.5,0.0,0.0,0.2,0.68,0.44
4,0.5,0.75,0.00,0.0,0.666667,0.625,0.625,0.250,0.50,0.450000,0.25,0.75,0.25,0.5,0.5,0.5,0.0,0.64,0.36


## Step 5 — Create the principal adult analysis sample without filtering on belief missingness

The metadata and constructed beliefs are combined and ordered by country and respondent ID, matching the ordering used for the historical Round 4 reference.

Respondents are retained when their age is at least 18 or when age is missing. **No respondent is removed because of the number of missing constructed beliefs.**

The following diagnostic columns are added:

- `n_belief_missing`: number of missing constructed beliefs;
- `n_belief_available`: number of available constructed beliefs;
- `cca_missingness_eligible`: `True` when no more than two beliefs are missing.

The last column is retained solely to reproduce the historical CCA-compatible subset for later validation.

In [5]:
curated_all = pd.concat(
    [metadata, beliefs],
    axis=1,
)
curated_all = curated_all.sort_values(
    ["cntry", "idno"],
    kind="stable",
).reset_index(drop=True)

analysis_sample = apply_analysis_sample_rule(
    curated_all,
    config.BELIEF_COLUMNS,
    age_column="agea",
    minimum_age=config.MINIMUM_AGE,
    cca_maximum_missing_beliefs=(
        config.CCA_MAXIMUM_MISSING_BELIEFS
    ),
).reset_index(drop=True)

assert len(analysis_sample) == config.EXPECTED_ANALYSIS_N, (
    f"Expected {config.EXPECTED_ANALYSIS_N:,} "
    f"principal-analysis rows, "
    f"found {len(analysis_sample):,}."
)
assert (
    analysis_sample["cntry"].nunique(dropna=True)
    == config.EXPECTED_ANALYSIS_COUNTRIES
)
assert (
    analysis_sample["agea"].ge(config.MINIMUM_AGE)
    | analysis_sample["agea"].isna()
).all()
assert (
    analysis_sample["n_belief_missing"]
    + analysis_sample["n_belief_available"]
).eq(config.EXPECTED_BELIEF_COUNT).all()

cca_subset = analysis_sample.loc[
    analysis_sample["cca_missingness_eligible"]
].copy()

assert len(cca_subset) == config.EXPECTED_CCA_N
assert (
    cca_subset["cntry"].nunique(dropna=True)
    == config.EXPECTED_CCA_COUNTRIES
)
assert cca_subset["n_belief_missing"].le(
    config.CCA_MAXIMUM_MISSING_BELIEFS
).all()

print(f"Raw ESS4 respondents: {len(raw):,}")
print(
    "Principal adult analysis respondents:",
    f"{len(analysis_sample):,}",
)
print(
    "Historical CCA-compatible respondents:",
    f"{len(cca_subset):,}",
)
print(
    "Adults retained despite more than two "
    "missing beliefs:",
    f"{(~analysis_sample['cca_missingness_eligible']).sum():,}",
)
print(
    "Countries retained:",
    analysis_sample["cntry"].nunique(dropna=True),
)

missingness_distribution = (
    analysis_sample["n_belief_missing"]
    .value_counts()
    .sort_index()
    .rename_axis("n_belief_missing")
    .rename("respondents")
    .to_frame()
)
missingness_distribution["percent"] = (
    100.0
    * missingness_distribution["respondents"]
    / len(analysis_sample)
)

display(missingness_distribution)

Raw ESS4 respondents: 56,752
Principal adult analysis respondents: 55,044
Historical CCA-compatible respondents: 45,268
Adults retained despite more than two missing beliefs: 9,776
Countries retained: 29


,respondents,percent
n_belief_missing,,
0,30126,54.730761
1,9561,17.369741
2,5581,10.139161
3,3068,5.573723
4,1993,3.620740
5,1373,2.494368
6,908,1.649589
7,662,1.202674
8,514,0.933798


## Step 6 — Inspect country counts and belief descriptives for the principal sample

These tables describe the full adult analysis sample. They are not restricted to respondents satisfying the historical CCA missingness threshold.

The respondent-level comparison with Van Noord's `df_ESS4.RData` is performed later using only `cca_missingness_eligible == True`, because the reference object represents the historical CCA-compatible sample.

In [6]:
country_counts = (
    analysis_sample
    .groupby(
        ["cntry", "country_name"],
        dropna=False,
    )
    .size()
    .rename("N")
    .reset_index()
)

analysis_belief_summary = summarise_beliefs(
    analysis_sample,
    config.BELIEF_COLUMNS,
)
analysis_belief_summary["N_total"] = len(
    analysis_sample
)
analysis_belief_summary["N_missing"] = (
    analysis_belief_summary["N_total"]
    - analysis_belief_summary["N_reproduced"]
)
analysis_belief_summary["percent_missing"] = (
    100.0
    * analysis_belief_summary["N_missing"]
    / analysis_belief_summary["N_total"]
)

print("Country-level principal sample sizes:")
display(country_counts)

print(
    "Belief-variable summary for the "
    "principal adult sample:"
)
display(analysis_belief_summary)

Country-level principal sample sizes:


,cntry,country_name,N
0,BE,Belgium,1682
1,BG,Bulgaria,2190
2,CH,Switzerland,1776
3,CY,Cyprus,1168
4,CZ,Czechia,1955
5,DE,Germany,2691
6,DK,Denmark,1550
7,EE,Estonia,1610
8,ES,Spain,2486
9,FI,Finland,2105


Belief-variable summary for the principal adult sample:


,belief_variable,N_reproduced,mean_reproduced,sd_reproduced,N_total,N_missing,percent_missing
0,left_right_identification,46413,0.519755,0.228265,55044,8631,15.680183
1,gender_inequality,53539,0.547302,0.260615,55044,1505,2.734176
2,anti_lgbt,51621,0.356798,0.320101,55044,3423,6.218661
3,euroscepticism,48543,0.464296,0.266711,55044,6501,11.810552
4,anti_immigration,51377,0.489335,0.276050,55044,3667,6.661943
5,anti_egalitarianism,53152,0.298446,0.210017,55044,1892,3.437250
6,benefits_harm_economy,48729,0.511923,0.226657,55044,6315,11.472640
7,benefits_harm_society,51915,0.427153,0.228826,55044,3129,5.684543
8,welfare_chauvinism,50826,0.568739,0.255371,55044,4218,7.662961
9,anti_economic_interventionism,52717,0.219460,0.161449,55044,2327,4.227527


## Step 7 — Create separate principal datasets with and without official weights

The two outputs contain exactly the same respondents, identifiers, demographics, belief values, missingness counts, and CCA-eligibility flags.

The with-weights version additionally includes `dweight`, `pspwght`, `pweight`, and `anweight` immediately after `blgetmg`. No observations are duplicated and no belief values are multiplied by weights.

In [7]:
analysis_without_weights, analysis_with_weights = (
    split_weighted_and_unweighted(
        analysis_sample,
        config.WEIGHT_COLUMNS,
        insert_after=config.WEIGHT_INSERT_AFTER,
    )
)

analysis_without_weights = (
    analysis_without_weights.loc[
        :, config.OUTPUT_COLUMNS_WITHOUT_WEIGHTS
    ].copy()
)
analysis_with_weights = (
    analysis_with_weights.loc[
        :, config.OUTPUT_COLUMNS_WITH_WEIGHTS
    ].copy()
)

assert (
    analysis_without_weights.shape
    == config.EXPECTED_WITHOUT_WEIGHTS_SHAPE
)
assert (
    analysis_with_weights.shape
    == config.EXPECTED_WITH_WEIGHTS_SHAPE
)

validate_weight_split(
    analysis_without_weights,
    analysis_with_weights,
    config.WEIGHT_COLUMNS,
    require_complete_weights=True,
)

print(
    "Without-weights shape:",
    analysis_without_weights.shape,
)
print(
    "With-weights shape:",
    analysis_with_weights.shape,
)
print(
    "The datasets differ only by the four "
    "official weight columns."
)

Without-weights shape: (55044, 35)
With-weights shape: (55044, 39)
The datasets differ only by the four official weight columns.


## Step 8 — Save the principal ESS4 datasets

The principal all-adult outputs are written to `data/processed/` using the filenames defined in `src/ess4_config.py`.

The old `ess4_cca_initial_beliefs_*.csv` files are not overwritten automatically. The historical CCA-subset validation file will be created by the revised validation notebook.

In [8]:
analysis_without_weights.to_csv(
    WITHOUT_WEIGHTS_PATH,
    index=False,
)
analysis_with_weights.to_csv(
    WITH_WEIGHTS_PATH,
    index=False,
)

print("Saved:")
print(" -", WITHOUT_WEIGHTS_PATH)
print(" -", WITH_WEIGHTS_PATH)

Saved:
 - /Users/karan/Desktop/SSM-MERC/polarization/Github/C72H-SimPol/data_curation/data/processed/ess4_beliefs_all_adults_without_weights.csv
 - /Users/karan/Desktop/SSM-MERC/polarization/Github/C72H-SimPol/data_curation/data/processed/ess4_beliefs_all_adults_with_weights.csv


## Step 9 — Read the saved files back and perform final integrity checks

Reading the outputs back from disk detects problems involving serialization, accidental index columns, schema order, respondent identity, missingness indicators, or survey weights.

In [9]:
saved_without = pd.read_csv(
    WITHOUT_WEIGHTS_PATH,
    low_memory=False,
)
saved_with = pd.read_csv(
    WITH_WEIGHTS_PATH,
    low_memory=False,
)

assert (
    saved_without.shape
    == config.EXPECTED_WITHOUT_WEIGHTS_SHAPE
)
assert (
    saved_with.shape
    == config.EXPECTED_WITH_WEIGHTS_SHAPE
)
assert list(saved_without.columns) == list(
    config.OUTPUT_COLUMNS_WITHOUT_WEIGHTS
)
assert list(saved_with.columns) == list(
    config.OUTPUT_COLUMNS_WITH_WEIGHTS
)

validate_weight_split(
    saved_without,
    saved_with,
    config.WEIGHT_COLUMNS,
    require_complete_weights=True,
)

assert saved_without["ess_unique_id"].is_unique
assert saved_with["ess_unique_id"].is_unique
assert saved_without["ess_unique_id"].equals(
    saved_with["ess_unique_id"]
)
assert (
    saved_without["n_belief_missing"]
    + saved_without["n_belief_available"]
).eq(config.EXPECTED_BELIEF_COUNT).all()

saved_cca_eligible = (
    saved_without["cca_missingness_eligible"]
    .astype("boolean")
)
assert int(saved_cca_eligible.sum()) == config.EXPECTED_CCA_N

print("ESS Round 4 curation completed successfully.")
print(f"Raw respondents: {config.EXPECTED_RAW_N:,}")
print(
    "Principal adult respondents:",
    f"{len(saved_without):,}",
)
print(
    "Historical CCA-compatible respondents:",
    f"{int(saved_cca_eligible.sum()):,}",
)
print(
    "Adults retained with more than two "
    "missing beliefs:",
    f"{int((~saved_cca_eligible).sum()):,}",
)
print(
    "Countries:",
    saved_without["cntry"].nunique(dropna=True),
)
print(
    "Belief variables:",
    len(config.BELIEF_COLUMNS),
)
print(
    "Without-weights columns:",
    saved_without.shape[1],
)
print(
    "With-weights columns:",
    saved_with.shape[1],
)
print("Missing official weights:")
print(
    saved_with.loc[:, config.WEIGHT_COLUMNS]
    .isna()
    .sum()
    .to_dict()
)

ESS Round 4 curation completed successfully.
Raw respondents: 56,752
Principal adult respondents: 55,044
Historical CCA-compatible respondents: 45,268
Adults retained with more than two missing beliefs: 9,776
Countries: 29
Belief variables: 19
Without-weights columns: 35
With-weights columns: 39
Missing official weights:
{'dweight': 0, 'pspwght': 0, 'pweight': 0, 'anweight': 0}
